# Week 4: Flood Prediction with Supervised Learning

## Executive Summary

During the first three weeks, the project established a complete data-science workflow using an algorithmic traffic dataset: preprocessing, exploratory analysis, and unsupervised clustering. That dataset was useful for validating the pipeline, but its uniform synthetic structure did not contain the behavioral skews needed for meaningful segmentation.

For the supervised-learning phase, the project therefore pivots to the verified Kaggle Flood Prediction Dataset, a real-world meteorological telemetry matrix. This transition supports the longer-term goal of forecasting severe urban waterlogging and identifying safer navigation routes from precipitation, elevation, location, and seasonal information.

In [31]:
import kagglehub

path = kagglehub.dataset_download("farahfirdausa/flood-prediction-dataset")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\raiti\.cache\kagglehub\datasets\farahfirdausa\flood-prediction-dataset\versions\1


In [32]:
import os
print(os.listdir(path))

['modis_flood_features_paling cleaning (1).csv']


In [33]:
import pandas as pd

df = pd.read_csv(f"{path}/modis_flood_features_paling cleaning (1).csv")
print(df.head())

         date         lon       lat  flooded  jrc_perm_water  precip_1d  \
0  2003-12-10  120.478677 -6.495942      1.0             1.0   9.645125   
1  2003-12-10  120.480923 -6.495942      1.0             1.0   9.645125   
2  2003-12-10  120.478677 -6.493697      0.0             1.0   9.645125   
3  2003-12-10  120.480923 -6.493697      1.0             0.0   9.645125   
4  2003-12-10  120.478677 -6.491451      0.0             1.0   9.645125   

   precip_3d        NDVI      NDWI  landcover  elevation     slope  \
0  28.935376  6190.93062 -0.051446       17.0        0.0  0.116685   
1  28.935376  6190.93062 -0.051446       17.0        1.0  0.839713   
2  28.935376  6190.93062  0.028399       17.0        0.0  0.466730   
3  28.935376  6190.93062  0.028399       17.0        4.0  0.939179   
4  28.935376  6190.93062 -0.002506       17.0        0.0  0.583403   

       aspect  upstream_area       TWI  target  
0  270.000000       0.031990  1.042098       0  
1  236.480074       0.031990  

In [34]:
print(df.shape)
print(df.isnull().sum())
print(df.dtypes)
print(df['target'].value_counts())

(1025801, 16)
date              0
lon               0
lat               0
flooded           0
jrc_perm_water    0
precip_1d         0
precip_3d         0
NDVI              0
NDWI              0
landcover         0
elevation         0
slope             0
aspect            0
upstream_area     0
TWI               0
target            0
dtype: int64
date               object
lon               float64
lat               float64
flooded           float64
jrc_perm_water    float64
precip_1d         float64
precip_3d         float64
NDVI              float64
NDWI              float64
landcover         float64
elevation         float64
slope             float64
aspect            float64
upstream_area     float64
TWI               float64
target              int64
dtype: object
target
0    979446
1     46355
Name: count, dtype: int64


## 1. Problem Definition and Dataset Inspection

This phase is a binary classification problem: predict a flood event (`target = 1`) versus no flood (`target = 0`) from spatial, temporal, and meteorological variables. The model is intended for an early-warning workflow, so recall is the critical metric: missing a real flood is more harmful than issuing an additional warning.

The initial inspection checks the dataset shape, missing values, data types, and target distribution. The raw matrix contains 1,025,801 observations, including 979,446 no-flood records and 46,355 flood records. This severe imbalance makes raw accuracy misleading because a classifier could mostly predict no flood and still appear successful.

In [35]:
# model will just predict "no flood" for almost everything and still get 95% accuracy, which is completely useless for a safety app
# undersample the majority class because we need all of minority class as it predicts flood

flood = df[df['target'] == 1]
no_flood = df[df['target'] == 0].sample(n=len(flood), random_state=42)

df_balanced = pd.concat([flood, no_flood]).sample(frac=1, random_state=42)
print(df_balanced['target'].value_counts())

target
1    46355
0    46355
Name: count, dtype: int64


## 2. Class Balancing

To prevent the classifier from learning the trivial rule "always predict no flood," the majority class is undersampled. All 46,355 flood observations are retained, and exactly 46,355 no-flood observations are sampled with a fixed random seed. The resulting `df_balanced` table has a 1:1 target ratio and is shuffled before training.

In [36]:
print(df_balanced[['flooded', 'target']].value_counts())

flooded  target
1.0      1         46355
0.0      0         43062
1.0      0          3293
Name: count, dtype: int64


In [37]:
df_balanced["month"] = pd.to_datetime(df_balanced["date"]).dt.month
df_balanced[['month']].value_counts()

month
12       23530
6        22211
5        14813
7        11920
2        10628
1         9608
Name: count, dtype: int64

## 3. Temporal Feature Engineering and Feature Selection

The raw date string is converted with `pd.to_datetime()` and reduced to a discrete `month` feature. This gives the model access to seasonal patterns, including monsoon timing and recurring precipitation cycles.

The final feature matrix focuses on the environmental drivers most relevant to flood risk: latitude (`lat`), longitude (`lon`), month, elevation, one-day precipitation (`precip_1d`), and three-day precipitation (`precip_3d`). The target is kept separately as `y`, so the model learns to predict flood occurrence from the selected inputs.

In [51]:
X = df_balanced[['lat', 'lon', 'month', 'elevation', 'precip_1d', 'precip_3d']]
y = df_balanced[['target']]

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Train-Test Split and Model Rationale

The balanced data is divided into training and unseen test partitions using an 80/20 split with `random_state=42`. XGBoost is selected because gradient-boosted decision trees can model non-linear interactions, such as the compounding risk of heavy three-day precipitation over low-elevation areas.

The `XGBClassifier` is configured with 100 boosting rounds, a maximum tree depth of 6, a learning rate of 0.1, and a fixed random state. The model is then fitted on `X_train` and `y_train`.

In [53]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,      # number of trees
    max_depth=6,           # how deep each tree grows
    learning_rate=0.1,     # how fast it learns
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [54]:
y_pred = model.predict(X_test)

In [55]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.88      0.92      9340
           1       0.89      0.96      0.92      9202

    accuracy                           0.92     18542
   macro avg       0.92      0.92      0.92     18542
weighted avg       0.92      0.92      0.92     18542

[[8207 1133]
 [ 381 8821]]


## 5. Evaluation and Performance Analysis

Predictions from the held-out test set are assessed with a classification report and confusion matrix. The supplied results show 92% overall accuracy, 0.96 recall for the flood class, and 0.89 flood precision. Recall is prioritized because the safety cost of a false negative is high: the model correctly identifies 8,821 of 9,202 actual flood events.

The confusion matrix, `[[8207, 1133], [381, 8821]]`, contains 381 false negatives and 1,133 false positives. This pattern reflects an intentionally safety-oriented classifier: it accepts additional warnings in order to miss as few real flood events as possible.

In [60]:
import joblib
joblib.dump(model, "flood_model.pkl")

['flood_model.pkl']

## 6. Model Serialization and Application Use

The trained estimator is serialized to `flood_model.pkl` with `joblib`, allowing a frontend or backend service to load the model without retraining. Because the classifier supports `predict_proba`, an application can display a percentage-based flood-risk score and apply configurable warning thresholds instead of exposing only a binary prediction.

In [57]:
print(df_balanced['target'].value_counts())
print(y_train.value_counts())

target
1    46355
0    46355
Name: count, dtype: int64
target
1         37153
0         37015
Name: count, dtype: int64


In [59]:
print(model.predict_proba([[-3.99, 120.01, 12, 5.0, 10.76, 75.37]]))

[[0.02569127 0.9743087 ]]


In [61]:
print(model.classes_)

[0 1]


## 7. Strengths, Limitations, and Next Steps

The main strength of this pipeline is its high flood recall: balancing the classes and using gradient boosting exposes the relationship between precipitation, terrain elevation, location, and seasonal timing. The serialized model also supports probability scoring for practical alert thresholds.

The main limitation is aggressive undersampling. More than 930,000 no-flood observations are discarded, so the model sees less variety in negative environmental conditions and its measured precision may not fully represent deployment conditions. In the next phase, SMOTE or another imbalance-aware strategy can be evaluated against this baseline. Training with more of the available terrain data may improve the 0.89 precision score and reduce the 1,133 false alarms, while threshold tuning can preserve the high recall needed for early warning.